# WikiRate API JSON import

This notebook loads `WIKIRATE_API_KEY` from `workbooks/.env`, connects to WikiRate, and pulls JSON from a configurable endpoint.

In [ ]:
from pathlib import Path
import json
import os
import re

import pandas as pd
import requests


BASE_URL = "https://wikirate.org"
ENV_PATHS = [Path(".env"), Path("workbooks/.env")]


def load_env_value(key: str, env_paths: list[Path] = ENV_PATHS) -> str:
    """Read one value from a simple .env file without requiring python-dotenv."""
    env_path = next((path for path in env_paths if path.exists()), None)
    if env_path is None:
        searched = ", ".join(str(path) for path in env_paths)
        raise FileNotFoundError(f"Could not find .env file. Searched: {searched}")

    for line in env_path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue

        name, value = line.split("=", 1)
        if name.strip() == key:
            return value.strip().strip('"').strip("'")

    raise KeyError(f"{key} was not found in {env_path.resolve()}")


WIKIRATE_API_KEY = os.getenv("WIKIRATE_API_KEY") or load_env_value("WIKIRATE_API_KEY")

session = requests.Session()
session.headers.update({
    "Accept": "application/json",
    "User-Agent": "wbs-esg-project/1.0",
})

print("WikiRate API key loaded.")

In [ ]:
def get_wikirate_json(endpoint: str, **params) -> dict | list:
    """Pull JSON from WikiRate. Use endpoints like '/Company.json' or '/Some_Card_Name.json'."""
    endpoint = endpoint if endpoint.startswith("/") else f"/{endpoint}"
    url = f"{BASE_URL}{endpoint}"

    request_params = {
        "api_key": WIKIRATE_API_KEY,
        **params,
    }

    response = session.get(url, params=request_params, timeout=30)
    response.raise_for_status()
    return response.json()


# Change this endpoint to the WikiRate card/data you want to import.
# WikiRate generally exposes page/card JSON by adding `.json` to the page/card name.
endpoint = "/Company.json"

data = get_wikirate_json(endpoint, limit=10)

print(type(data))
print(json.dumps(data, indent=2)[:2000])

In [ ]:
# Optional: flatten JSON into a DataFrame if the response shape is list-like.
if isinstance(data, list):
    df = pd.json_normalize(data)
elif isinstance(data, dict):
    list_values = [value for value in data.values() if isinstance(value, list)]
    df = pd.json_normalize(list_values[0]) if list_values else pd.json_normalize(data)
else:
    df = pd.DataFrame()

df.head()

In [ ]:
# Optional: save raw JSON for reproducibility, without storing the API key in returned URLs.
output_path = Path("wikirate_raw_response.json")
redacted_json = re.sub(r"api_key=[^&\"]+", "api_key=REDACTED", json.dumps(data, indent=2))
output_path.write_text(redacted_json, encoding="utf-8")
output_path